In [43]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from pycds import Variable
from crmprtd.infer import infer
import os
import pickle
import logging
from crmprtd.more_itertools import tap, log_progress
from crmprtd.align import align
from crmprtd.insert import insert
from pprint import pprint
from crmprtd import Row

log = logging.getLogger(__name__)

engine = sa.create_engine("postgresql://tongli1997@dbtest04.pcic.uvic.ca:5432/crmp", echo=False)
session = Session(engine)
session.rollback()

query = sa.text("SELECT * FROM meta_vars WHERE network_id = :nid")
result = session.execute(query, {"nid": 11})
# for row in result:
#     print(row)

query = sa.text("SELECT net_var_name, unit FROM meta_vars WHERE network_id = :nid")
db_unit = session.execute(query, {"nid": 11})
db_var_unit = db_unit.all()
db_var_unit

# results2 = session.query(Variable).filter(Variable.network_id == 11)
# test2 = results2.all()
# pprint(test2)

# names = [v.name for v in test2]
# print(names)
# print(len(names))


[('RHx', '%'),
 ('WindSpeedms', 'm s-1'),
 ('Wind_n', 'm s-1'),
 ('RH', '%'),
 ('DewPtC', 'celsius'),
 ('Pressurembar', 'millibar'),
 ('Rainmm', 'mm'),
 ('Tm', 'celsius'),
 ('Tx', 'celsius'),
 ('Tn', 'celsius'),
 ('TempC', 'celsius'),
 ('WindDirection', 'degree'),
 ('RHn', '%'),
 ('GustSpeedms', 'm s-1'),
 ('Tx_Climatology', 'celsius'),
 ('SolarRadiationWm', 'W m-2'),
 ('Wetness', '%'),
 ('Tn_Climatology', 'celsius'),
 ('T_mean_Climatology', 'celsius'),
 ('Precip_Climatology', 'mm'),
 ('Soil_Temp', 'celsius'),
 ('Water_Content_m3_m3_15_cm', 'm3/m3'),
 ('Water_Content_m3_m3_30_cm', 'm3/m3'),
 ('Water_Content_m3_m3_5_cm', 'm3/m3'),
 ('Snow_Depth', 'cm'),
 ('Wetness_20cm', '%')]

#### Some test commands using sqlalchemy

In [42]:
# import os
# print(os.environ.get("HOME"))

### Check which tables exist
from sqlalchemy import inspect

inspector = inspect(engine)
tables = inspector.get_table_names()
print("Tables in the database:", tables)


###  See columns and their types
columns = inspector.get_columns("meta_vars")
for col in columns:
    print(col["name"], col["type"])

## Count records
count = session.execute(sa.text("SELECT COUNT(*) FROM meta_vars")).scalar()
print("Number of rows:", count)

### Get distinct values
result = session.execute(sa.text("SELECT DISTINCT network_id FROM meta_vars"))
print(result.fetchall())

#### Run joins
sql = """
SELECT v.net_var_name, n.network_name
FROM crmp.meta_vars v
JOIN crmp.meta_network n ON v.network_id = n.network_id
WHERE v.network_id = 11
LIMIT 5
"""
for row in session.execute(sa.text(sql)):
    print(row)

Tables in the database: ['spatial_ref_sys', 'meta_station', 'obs_raw', 'meta_network', 'meta_native_flag', 'obs_raw_native_flags', 'alembic_version', 'meta_network_geoserver', 'meta_alias', 'meta_contact', 'meta_history_hx', 'meta_sensor', 'test_mv', 'time_bounds', 'obs_raw_hx', 'meta_vars', 'meta_history', 'meta_pcic_flag', 'meta_vars_hx', 'obs_raw_pcic_flags', 'meta_network_hx', 'meta_station_hx', 'stats_station_var', 'obs_derived_values', 'climo_period_hx', 'climo_period', 'climo_station_hx', 'climo_station', 'climo_stn_x_hist_hx', 'climo_stn_x_hist', 'climo_variable_hx', 'climo_variable', 'climo_value_hx', 'climo_value']
vars_id INTEGER
network_id INTEGER
unit VARCHAR(64)
precision NUMERIC
standard_name VARCHAR(64)
cell_method VARCHAR(64)
long_description VARCHAR(256)
net_var_name CITEXT
display_name VARCHAR(256)
short_name VARCHAR(256)
mod_time TIMESTAMP
mod_user VARCHAR(64)
Number of rows: 350
[(39,), (11,), (19,), (36,), (34,), (21,), (40,), (14,), (3,), (17,), (20,), (22,), (13

In [50]:
output_folder = '/workspaces/crmprtd/rows_output/'

# file_name = 'rows_output_AtlSch.pickle'
file_name = 'updated_rows_output_Barren.pickle'

file_path = os.path.join(output_folder, file_name)

# Load the pickle file
with open(file_path, 'rb') as f:
    rows = pickle.load(f)

# pprint(rows)

variable_unit_fern = [(r.variable_name, r.unit) for r in rows]
print(variable_unit_fern) 
print(len(variable_unit_fern)) 



[('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('Pressurembar', 'mbar'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('TempC', '°C'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('DewPtC', '°C'), ('WindSpeedms', 'm/s'), ('WindSpeedms', 'm/s'), ('WindSpeedms', 'm/s'), ('WindS

In [54]:
db_unit_map = {name: unit for name, unit in db_var_unit}

db_unit_map

{'RHx': '%',
 'WindSpeedms': 'm s-1',
 'Wind_n': 'm s-1',
 'RH': '%',
 'DewPtC': 'celsius',
 'Pressurembar': 'millibar',
 'Rainmm': 'mm',
 'Tm': 'celsius',
 'Tx': 'celsius',
 'Tn': 'celsius',
 'TempC': 'celsius',
 'WindDirection': 'degree',
 'RHn': '%',
 'GustSpeedms': 'm s-1',
 'Tx_Climatology': 'celsius',
 'SolarRadiationWm': 'W m-2',
 'Wetness': '%',
 'Tn_Climatology': 'celsius',
 'T_mean_Climatology': 'celsius',
 'Precip_Climatology': 'mm',
 'Soil_Temp': 'celsius',
 'Water_Content_m3_m3_15_cm': 'm3/m3',
 'Water_Content_m3_m3_30_cm': 'm3/m3',
 'Water_Content_m3_m3_5_cm': 'm3/m3',
 'Snow_Depth': 'cm',
 'Wetness_20cm': '%'}

In [56]:
# Step 1. Make a dict: variable name → DB unit
db_unit_map = {name: unit for name, unit in db_var_unit}

# Step 2. Replace Fern units with DB units if available
variable_unit_aligned = []
for name, unit in variable_unit_fern:
    new_unit = db_unit_map.get(name, unit)  # use DB unit if found, else keep original
    variable_unit_aligned.append((name, new_unit))

# Step 3. Optional: check differences
mismatched = [
    (name, old_unit, db_unit_map[name])
    for name, old_unit in variable_unit_fern
    if name in db_unit_map and db_unit_map[name] != old_unit
]

print("✅ Aligned units (first few):")
print(variable_unit_aligned)

print("\n⚠️ Variables with unit differences:")
from pprint import pprint
pprint(mismatched)

✅ Aligned units (first few):
[('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Rainmm', 'mm'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('Pressurembar', 'millibar'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('TempC', 'celsius'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('RH', '%'), ('DewPtC', 'celsius'), ('DewPtC', 'celsius'), ('DewPtC', 'celsius'), ('DewPtC', 'celsius'), ('DewPtC', 'celsius'), ('DewPtC', 'celsius'), ('

In [83]:
unique_aligned = {}
for name, unit in variable_unit_aligned:
    if name not in unique_aligned:
        unique_aligned[name] = unit

print("✅ Aligned units (first few unique):")
for name, unit in list(unique_aligned.items()):
    print(f"{name}: {unit}")
    
unique_mismatched = {}
for name, old_unit, new_unit in mismatched:
    if name not in unique_mismatched:
        unique_mismatched[name] = (old_unit, new_unit)

print("\n⚠️ Unique variables with unit differences:")
from pprint import pprint
pprint(list(unique_mismatched.items()))

✅ Aligned units (first few unique):
Rainmm: mm
Pressurembar: millibar
TempC: celsius
RH: %
DewPtC: celsius
WindSpeedms: m s-1
GustSpeedms: m s-1
WindDirection: degree
SolarRadiationWm: W m-2
Wetness: %
Snow_Depth: cm
Soil_Temp: celsius

⚠️ Unique variables with unit differences:
[('Pressurembar', ('mbar', 'millibar')),
 ('TempC', ('°C', 'celsius')),
 ('DewPtC', ('°C', 'celsius')),
 ('WindSpeedms', ('m/s', 'm s-1')),
 ('GustSpeedms', ('m/s', 'm s-1')),
 ('WindDirection', ('ø', 'degree')),
 ('SolarRadiationWm', ('W/m²', 'W m-2')),
 ('Soil_Temp', ('°C', 'celsius'))]


In [60]:
rows

[Row(time=datetime.datetime(2021, 7, 9, 12, 0), val=nan, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 13, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 14, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 15, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 16, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 17, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barr

In [64]:
variable_unit_aligned

[('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC'

In [66]:

updated_rows = []

for row in rows:
    # Copy row to avoid modifying the original
    new_row = row.copy()
    
    # Update unit if variable_name exists in variable_unit_aligned
    if row['variable_name'] in variable_unit_aligned:
        new_row['unit'] = variable_unit_aligned[row['variable_name']]
    
    updated_rows.append(new_row)

AttributeError: 'Row' object has no attribute 'copy'

In [69]:
variable_unit_aligned

[('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Rainmm', 'mm'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('Pressurembar', 'millibar'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('TempC', 'celsius'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('RH', '%'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC', 'celsius'),
 ('DewPtC'

In [73]:
variable_unit_aligned.get(new_var, r.unit)

AttributeError: 'list' object has no attribute 'get'

In [77]:
updated_rows = []
variable_unit_aligned_dict = dict(variable_unit_aligned)
for r in rows:
    # Keep the original variable name or assign new_var if you already changed names
    new_var = r.variable_name  # or some logic to update variable_name
    
    # Update unit if there is a mapped value in variable_unit_aligned
    new_unit = variable_unit_aligned_dict.get(new_var, r.unit)
    print(new_var, new_unit)

    # Create a new Row with updated unit
    updated_rows.append(
        Row(
            time=r.time,
            val=r.val,
            variable_name=new_var,
            unit=new_unit,  # <-- updated unit here
            network_name=r.network_name,
            station_id=r.station_id,
            lat=r.lat,
            lon=r.lon
        )
    )

Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Rainmm mm
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
Pressurembar millibar
TempC celsius
TempC celsius
TempC celsius
TempC celsius
TempC celsius
TempC celsius
TempC celsius
TempC celsius
TempC celsius
TempC celsius
RH %
RH %
RH %
RH %
RH %
RH %
RH %
RH %
RH %
RH %
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
DewPtC celsius
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
WindSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-1
GustSpeedms m s-

In [79]:
updated_rows

[Row(time=datetime.datetime(2021, 7, 9, 12, 0), val=nan, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 13, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 14, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 15, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 16, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barren', lat=54.510444, lon=-126.614341),
 Row(time=datetime.datetime(2021, 7, 9, 17, 0), val=0.0, variable_name='Rainmm', unit='mm', network_name='FLNRO-FERN', station_id='Barr

In [80]:
len(updated_rows)

120

In [78]:
updated_rows

output_folder = '/workspaces/crmprtd/rows_output/'

out_file = os.path.join(output_folder, f'updated_rows_with_unit_output_Barren.pickle')
with open(out_file, 'wb') as f:
    pickle.dump(updated_rows, f)